In [180]:
import pandas as pd
import numpy as np

In [181]:
# Load raw dataset
raw_data = pd.read_csv("university_raw_data.csv")

print("Raw data shape:", raw_data.shape)

Raw data shape: (4168, 21)


In [182]:
clean_data = raw_data[
    ~((raw_data["Source"] == "QS") & (raw_data["Rank"] == "Reporter"))
].copy()

In [183]:
import re

def convert_rank(value):
    value = str(value).strip()
    value = value.replace("â€“", "-").replace("–", "-")
    
    if "+" in value:
        return float(value.replace("+", ""))
    
    if "-" in value:
        start, end = value.split("-")
        return (float(start) + float(end)) / 2
    
    return float(value)

clean_data["Rank_Numeric"] = clean_data["Rank"].apply(convert_rank)

In [184]:
clean_data["Rank_Normalized"] = (
    clean_data["Rank_Numeric"].max() - clean_data["Rank_Numeric"]
) / (
    clean_data["Rank_Numeric"].max() - clean_data["Rank_Numeric"].min()
)

In [185]:
def convert_score(value):
    value = str(value).strip()
    value = value.replace("â€“", "-").replace("–", "-")
    
    if value == "-" or value == "nan":
        return np.nan
    
    if "-" in value:
        start, end = value.split("-")
        return (float(start) + float(end)) / 2
    
    return float(value)

clean_data["Overall_Score_Numeric"] = clean_data["Overall Score"].apply(convert_score)

In [186]:
clean_data["University_Clean"] = (
    clean_data["University"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [187]:
clean_data["Country_Clean"] = (
    clean_data["Country"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [188]:
country_mapping = {
    "Brunei": "Brunei Darussalam",
    "China": "China (Mainland)",
    "Hong Kong": "Hong Kong SAR",
    "Iran": "Iran, Islamic Republic of",
    "Macau": "Macao SAR",
    "Russia": "Russian Federation",
    "Palestine": "Palestinian Territory, Occupied"
}

clean_data["Country_Clean"] = clean_data["Country_Clean"].replace(country_mapping)

In [189]:
clean_data["International Students %"] = pd.to_numeric(
    clean_data["International Students %"],
    errors="coerce"
)

In [190]:
the_mask = clean_data["Source"] == "THE"

the_overall_median = clean_data.loc[
    the_mask, "Overall_Score_Numeric"
].median()

clean_data.loc[the_mask, "Overall_Score_Numeric"] = (
    clean_data.loc[the_mask, "Overall_Score_Numeric"]
    .fillna(the_overall_median)
)

print("THE Overall Score median:", the_overall_median)

THE Overall Score median: 34.55


In [191]:
# Missing percentage for every column
missing_percent = final_cleaned.isnull().mean() * 100

print(missing_percent)

print("\nColumns with more than 2% missing:")
print(missing_percent[missing_percent > 2])

Rank                                    0.0
University                              0.0
Overall Score                           0.0
Teaching Score                          0.0
Research Score                          0.0
Citations Score                         0.0
Industry Income Score                   0.0
International Outlook Score             0.0
Country                                 0.0
Student Staff Ratio                     0.0
International Students %                0.0
Source                                  0.0
Academic Reputation Score               0.0
Employer Reputation Score               0.0
Faculty Student Score                   0.0
Citations per Faculty Score             0.0
International Faculty Score             0.0
International Students Score            0.0
International Research Network Score    0.0
Employment Outcomes Score               0.0
Sustainability Score                    0.0
Rank_Numeric                            0.0
Rank_Normalized                 

In [192]:
final_cleaned = clean_data.copy()

performance_cols = [
    "Teaching Score",
    "Research Score",
    "Citations Score",
    "Industry Income Score",
    "International Outlook Score",
    "Student Staff Ratio",
    "International Students %",
    "Academic Reputation Score",
    "Employer Reputation Score",
    "Faculty Student Score",
    "Citations per Faculty Score",
    "International Faculty Score",
    "International Students Score",
    "International Research Network Score",
    "Employment Outcomes Score",
    "Sustainability Score"
]

# Step 1: Fill using source-wise median
for col in performance_cols:
    final_cleaned[col] = final_cleaned.groupby("Source")[col].transform(
        lambda x: x.fillna(x.median())
    )

# Step 2: Fill remaining NaN using overall column median
for col in performance_cols:
    final_cleaned[col] = final_cleaned[col].fillna(
        final_cleaned[col].median()
    )

C:\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Python313\Lib\site-packag

In [193]:
final_cleaned.to_csv("university_cleaned.csv", index=False)